# Smart Energy Consumption Prediction System
## Data Engineering & Preprocessing
### 01 - Dataset Understanding

**Dataset:** Individual Household Electric Power Consumption

**Objective:** Understand the structure, columns, data types, target variable, timestamp fields, and basic characteristics of the raw dataset before performing any cleaning.

In [16]:
# Import pandas for data loading and analysis
import pandas as pd

In [17]:
# Find the raw dataset path
from pathlib import Path

possible_paths = [
    Path("data/raw/household_power_consumption.txt"),
    Path("../data/raw/household_power_consumption.txt")
]

file_path = next((path for path in possible_paths if path.exists()), None)

if file_path is None:
    raise FileNotFoundError("Raw dataset not found.")

print("Dataset path:", file_path)

Dataset path: ..\data\raw\household_power_consumption.txt


In [18]:
# Load the raw semicolon-separated dataset
df = pd.read_csv(
    file_path,
    sep=";",
    na_values="?",
    low_memory=False
)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [19]:
# Display the first five records
df.head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [20]:
# Display the last five records
df.tail()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
2075254,26/11/2010,20:58:00,0.946,0.0,240.43,4.0,0.0,0.0,0.0
2075255,26/11/2010,20:59:00,0.944,0.0,240.00,4.0,0.0,0.0,0.0
2075256,26/11/2010,21:00:00,0.938,0.0,239.82,3.8,0.0,0.0,0.0
2075257,26/11/2010,21:01:00,0.934,0.0,239.70,3.8,0.0,0.0,0.0
2075258,26/11/2010,21:02:00,0.932,0.0,239.55,3.8,0.0,0.0,0.0


In [21]:
# Check the number of rows and columns
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 2075259
Number of columns: 9


In [22]:
# List all dataset columns
for column in df.columns:
    print(column)

Date
Time
Global_active_power
Global_reactive_power
Voltage
Global_intensity
Sub_metering_1
Sub_metering_2
Sub_metering_3


In [23]:
# Inspect column types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   str    
 1   Time                   str    
 2   Global_active_power    float64
 3   Global_reactive_power  float64
 4   Voltage                float64
 5   Global_intensity       float64
 6   Sub_metering_1         float64
 7   Sub_metering_2         float64
 8   Sub_metering_3         float64
dtypes: float64(7), str(2)
memory usage: 142.5 MB


In [24]:
# Display the current data type of each column
df.dtypes

Date                         str
Time                         str
Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object

In [25]:
# Generate summary statistics for numeric columns
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Global_active_power,2049280.0,1.091615,1.057294,0.076,0.308,0.602,1.528,11.122
Global_reactive_power,2049280.0,0.123714,0.112722,0.000,0.048,0.100,0.194,1.390
Voltage,2049280.0,240.839858,3.239987,223.200,238.990,241.010,242.890,254.150
Global_intensity,2049280.0,4.627759,4.444396,0.200,1.400,2.600,6.400,48.400
Sub_metering_1,2049280.0,1.121923,6.153031,0.000,0.000,0.000,0.000,88.000
Sub_metering_2,2049280.0,1.298520,5.822026,0.000,0.000,0.000,1.000,80.000
Sub_metering_3,2049280.0,6.458447,8.437154,0.000,0.000,1.000,17.000,31.000


In [26]:
# Count missing values in each column
df.isna().sum()

Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64

In [27]:
# Calculate missing counts and percentages
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})

missing_summary

,missing_count,missing_percentage
Date,0,0.00
Time,0,0.00
Global_active_power,25979,1.25
Global_reactive_power,25979,1.25
Voltage,25979,1.25
Global_intensity,25979,1.25
Sub_metering_1,25979,1.25
Sub_metering_2,25979,1.25
Sub_metering_3,25979,1.25


In [28]:
# Count fully duplicated records
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [29]:
# Count unique values in each column
df.nunique()

Date                     1442
Time                     1440
Global_active_power      4186
Global_reactive_power     532
Voltage                  2837
Global_intensity          221
Sub_metering_1             88
Sub_metering_2             81
Sub_metering_3             32
dtype: int64

## Timestamp and Target Identification

### Timestamp Fields

The raw dataset stores the timestamp in two separate columns:

- `Date`
- `Time`

These fields will later be combined and converted into a proper datetime column during the datetime handling stage.

### Primary Target Variable

The selected primary prediction target is:

**`Global_active_power`**

It represents the household's global minute-averaged active power.

The remaining electrical measurements will be retained as supporting variables.

## Initial Dataset Understanding Summary

- The dataset contains household electricity measurements recorded over time.
- The raw dataset contains 9 columns.
- Date and time are stored separately.
- `Global_active_power` is selected as the main prediction target.
- The remaining electrical variables can be used as supporting features.
- Missing values were inspected but not handled.
- Duplicate records were inspected but not removed.
- No cleaning or transformation was performed in this notebook.

### Next Step

Perform detailed missing-value analysis and determine an appropriate treatment strategy.